# Heart Disease Risk Classifier — Feature Engineering & Model Training

MLOps Assignment 01 (AIMLCZG523) — Tasks 2 & 3: Feature Engineering, Model Development, and Experiment Tracking with MLflow.

This notebook mirrors `src/train.py` in exploratory/notebook form: it builds a reusable preprocessing pipeline, trains + tunes Logistic Regression and Random Forest classifiers, logs everything to MLflow, and saves the best model.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from preprocessing import load_raw, clean_and_binarize, build_preprocessor, get_feature_target_split

## 1. Load and prepare data

In [2]:
raw = load_raw('../data/raw/heart_disease_uci_raw.csv')
df = clean_and_binarize(raw)
X, y = get_feature_target_split(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape

((242, 13), (61, 13))

## 2. Set up MLflow experiment tracking

In [3]:
mlflow.set_tracking_uri('sqlite:///../mlflow.db')
mlflow.set_experiment('heart-disease-classification')

<Experiment: artifact_location='/home/claude/heart-disease-mlops/src/mlruns/1', creation_time=1783144961428, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783144961428, lifecycle_stage='active', name='heart-disease-classification', tags={}, trace_location=None, workspace='default'>

## 3. Train & tune Logistic Regression

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_pipe = Pipeline([
    ('preprocess', build_preprocessor()),
    ('clf', LogisticRegression(max_iter=2000, random_state=42))
])
logreg_grid = {'clf__C': [0.01, 0.1, 1, 10], 'clf__solver': ['lbfgs']}

with mlflow.start_run(run_name='logistic_regression_notebook'):
    search = GridSearchCV(logreg_pipe, logreg_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    search.fit(X_train, y_train)
    best_logreg = search.best_estimator_

    y_pred = best_logreg.predict(X_test)
    y_proba = best_logreg.predict_proba(X_test)[:, 1]
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
    }
    mlflow.log_params(search.best_params_)
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(best_logreg, artifact_path='model', serialization_format='pickle')

    print('Best params:', search.best_params_)
    print('Test metrics:', metrics)

2026/07/04 06:12:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/07/04 06:12:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Best params: {'clf__C': 1, 'clf__solver': 'lbfgs'}
Test metrics: {'accuracy': 0.8852459016393442, 'precision': 0.8387096774193549, 'recall': 0.9285714285714286, 'f1_score': 0.8813559322033898, 'roc_auc': 0.9664502164502166}


## 4. Train & tune Random Forest

In [5]:
rf_pipe = Pipeline([
    ('preprocess', build_preprocessor()),
    ('clf', RandomForestClassifier(random_state=42))
])
rf_grid = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [None, 5, 10],
    'clf__min_samples_split': [2, 5],
}

with mlflow.start_run(run_name='random_forest_notebook'):
    search_rf = GridSearchCV(rf_pipe, rf_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
    search_rf.fit(X_train, y_train)
    best_rf = search_rf.best_estimator_

    y_pred_rf = best_rf.predict(X_test)
    y_proba_rf = best_rf.predict_proba(X_test)[:, 1]
    metrics_rf = {
        'accuracy': accuracy_score(y_test, y_pred_rf),
        'precision': precision_score(y_test, y_pred_rf),
        'recall': recall_score(y_test, y_pred_rf),
        'f1_score': f1_score(y_test, y_pred_rf),
        'roc_auc': roc_auc_score(y_test, y_proba_rf),
    }
    mlflow.log_params(search_rf.best_params_)
    mlflow.log_metrics(metrics_rf)
    mlflow.sklearn.log_model(best_rf, artifact_path='model', serialization_format='pickle')

    print('Best params:', search_rf.best_params_)
    print('Test metrics:', metrics_rf)

2026/07/04 06:12:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/07/04 06:12:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Best params: {'clf__max_depth': None, 'clf__min_samples_split': 5, 'clf__n_estimators': 300}
Test metrics: {'accuracy': 0.8688524590163934, 'precision': 0.8333333333333334, 'recall': 0.8928571428571429, 'f1_score': 0.8620689655172413, 'roc_auc': 0.946969696969697}


## 5. Compare models and pick the winner

In [6]:
comparison = pd.DataFrame([
    {'model': 'logistic_regression', **metrics},
    {'model': 'random_forest', **metrics_rf},
])
comparison

,model,accuracy,precision,recall,f1_score,roc_auc
0,logistic_regression,0.885246,0.838710,0.928571,0.881356,0.96645
1,random_forest,0.868852,0.833333,0.892857,0.862069,0.94697


## 6. Persist the best model for the serving API

We save the *entire* pipeline (preprocessing + classifier) as a single artifact, so the FastAPI service never needs to duplicate preprocessing logic.

In [7]:
best_model, best_name = (best_logreg, 'logistic_regression') if metrics['roc_auc'] >= metrics_rf['roc_auc'] else (best_rf, 'random_forest')

joblib.dump(best_model, '../models/best_model.joblib')
joblib.dump({'model_name': best_name, 'feature_columns': list(X.columns)}, '../models/model_metadata.joblib')
print(f'Saved best model: {best_name}')

Saved best model: logistic_regression


### Notes
- Run `mlflow ui --backend-store-uri sqlite:///../mlflow.db` from the `notebooks/` directory's parent to browse all experiment runs in the MLflow UI.
- `src/train.py` is the script-based, CI/CD-friendly equivalent of this notebook and is what the GitHub Actions pipeline actually executes.